# 04 — Benchmarking

**Purpose:** Contextualize creator performance using real public benchmarks from Humanz and Ubiquitous case studies.

This notebook:
- Loads the real benchmark table (6 campaigns, all source-attributed)
- Compares creator dataset metrics against industry campaign benchmarks
- Provides context for what partnerships teams should reference in client conversations

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go

from ingest_real_data import load_benchmarks, load_scored_channels
from utils import set_plot_style, COLORS, format_number, format_currency

set_plot_style()

## 1. Real Benchmark Table

Every row below comes from a publicly available case study. Source URLs are included for verification.

In [ ]:
benchmarks = load_benchmarks()
print(f'{len(benchmarks)} real benchmark campaigns loaded')
print(f'Agencies: {benchmarks["agency"].unique().tolist()}')
print(f'All rows have source URLs: {benchmarks["source_url"].notna().all()}')
print()
benchmarks[['campaign', 'brand', 'agency', 'platform', 'views', 'impressions', 'cpm_usd', 'engagements', 'source_url']]

## 2. CPM Benchmarks

In [ ]:
cpm_data = benchmarks[benchmarks['cpm_usd'].notna()].sort_values('cpm_usd')

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cpm_data['campaign'], cpm_data['cpm_usd'],
               color=[COLORS['success'] if c < 5 else COLORS['accent'] if c < 10 else COLORS['danger']
                      for c in cpm_data['cpm_usd']], alpha=0.8)

for bar, val in zip(bars, cpm_data['cpm_usd']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'${val:.2f}', va='center', fontweight='bold')

ax.set_xlabel('CPM (USD)')
ax.set_title('Real Campaign CPM Benchmarks\n(from public Humanz / Ubiquitous case studies)')
ax.set_xlim(0, 14)
plt.tight_layout()
plt.savefig('../dashboard/screenshots/cpm_benchmarks.png')
plt.show()

print(f'CPM range: ${cpm_data["cpm_usd"].min():.2f} – ${cpm_data["cpm_usd"].max():.2f}')
print(f'Average CPM: ${cpm_data["cpm_usd"].mean():.2f}')

## 3. Views and Engagement Benchmarks

In [ ]:
views_data = benchmarks[benchmarks['views'].notna()].sort_values('views', ascending=False)

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(views_data['campaign'], views_data['views'] / 1e6,
               color=COLORS['primary'], alpha=0.8)

for bar, val in zip(bars, views_data['views']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            format_number(val), va='center', fontweight='bold')

ax.set_xlabel('Views (Millions)')
ax.set_title('Real Campaign Views Benchmarks')
plt.tight_layout()
plt.show()

# Engagement
eng_data = benchmarks[benchmarks['engagements'].notna()]
print('\nEngagement benchmarks:')
for _, row in eng_data.iterrows():
    print(f"  {row['campaign']} ({row['brand']}): {format_number(row['engagements'])} engagements")

## 4. Creator Dataset vs Industry Benchmarks

How do the channels in our dataset compare to real campaign performance benchmarks?

In [ ]:
channels = load_scored_channels()

# Compare 30-day views to campaign view benchmarks
ch_30d_views = channels['views_last_30d'].dropna()

print('=== Creator Dataset: 30-Day Views Distribution ===')
print(f'  Channels with 30d data: {len(ch_30d_views)}')
print(f'  Median 30d views: {format_number(ch_30d_views.median())}')
print(f'  Mean 30d views: {format_number(ch_30d_views.mean())}')
print(f'  Top 10% threshold: {format_number(ch_30d_views.quantile(0.9))}')
print()
print('=== Campaign Benchmarks (for context) ===')
for _, row in views_data.iterrows():
    print(f'  {row["campaign"]}: {format_number(row["views"])} total campaign views')
print()
print('Note: Campaign views are total across multiple creators.')
print('Individual channel 30-day views are not directly comparable to campaign totals.')
print('This comparison provides directional context only.')

In [ ]:
# Estimated earnings context
ch_earnings = channels['est_monthly_earnings_mid'].dropna()
print('=== Creator Dataset: Estimated Monthly Earnings ===')
print(f'  Channels with earnings data: {len(ch_earnings)}')
print(f'  Median monthly earnings: {format_currency(ch_earnings.median())}')
print(f'  Mean monthly earnings: {format_currency(ch_earnings.mean())}')
print(f'  Top 10% threshold: {format_currency(ch_earnings.quantile(0.9))}')
print()
print('Note: These are estimated earnings from the dataset, not verified actuals.')
print('They provide a rough order-of-magnitude reference for partnerships pricing discussions.')

---

## Summary for Partnerships Teams

**What benchmarks tell us:**
- Real campaign CPMs range from **$1.47 to $11.00** across TikTok/Instagram campaigns
- Top campaigns achieve **5M–9M+ views** with meaningful engagement
- Engagement multipliers of **6x industry standard** are achievable with the right creator-brand fit

**What our creator dataset shows:**
- Top YouTube channels generate significant 30-day views, but individual channel metrics are not directly comparable to multi-creator campaign totals
- Estimated earnings provide directional pricing context

**Important caveat:** This dataset does not contain ad spend, click, or conversion data. True CPM and ROI calculations require integration with campaign management platforms.